# `protein_based_graph` — quantile × criterion sweep (display only)

Ranking molecules by binding-profile information before the message-passing
**quantile filter**, then measuring the approved pipeline GNN across coverage
quantiles. **This notebook only displays results.** All numbers are produced by

    scripts/modeling/train/run_quantile_criteria_sweep.py

which runs the exact ensemble-pipeline GNN (`GnnSignedExtractor` → `_SignedSage`,
5-model bag, emit=prot) with the shared 7 criteria (`orbind.mol_selection`) and
writes the CSVs read below. Re-run that script to refresh the data; nothing here
trains anything.

In [ ]:
import sys, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as _t

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from orbind.mol_selection import CRITERIA

# --- what run_quantile_criteria_sweep.py produced ---
REGIME    = 'inductive_molecule_v5'          # pipeline regime key (transductive -> 'transductive')
SEEDS     = [42, 43, 44, 45, 46]
QUANTILES = [50, 80, 85, 90, 95, 99]
N_MODELS  = 5
METR = ['AUROC', 'AUPRC', 'MCC', 'F1']
CACHE  = ROOT / 'results/graph/full_full/v6/protein_based_graph'
GBCSV  = CACHE / f'metrics_{REGIME}__chemberta_77m_m2or.csv'                  # ChemBERTa-node bag
GBGCSV = CACHE / f'metrics_{REGIME}__gin_supervised_contextpred_all_m2or.csv' # GIN-node bag

# --- styling: 7 criteria (categorical) + boost reference line ---
COL = {'coverage': '#2a78d6', 'balance_bits': '#eb6834', 'entropy_bits': '#1baf7a',
       'disc_pairs': '#eda100', 'idf_coverage': '#e87ba4', 'composite': '#008300',
       'greedy_pair_cover': '#4a3aa7', 'boost_full': '#0b0b0b'}
LABEL = {'boost_full': 'boost (full)'}
ORDER = list(CRITERIA) + ['boost_full']

def sty(c):
    if c == 'boost_full':
        return dict(ls='-', lw=2.6, marker='s', zorder=6)
    return dict(ls='-', lw=2.0, marker='o', zorder=3)

def mean_ci(vals):
    v = np.asarray(vals, float); n = len(v)
    if n < 2:
        return (v.mean() if n else np.nan), 0.0
    return v.mean(), _t.ppf(0.975, n - 1) * v.std(ddof=1) / np.sqrt(n)

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('display setup ready; reads:', GBCSV.name, '|', GBGCSV.name)

## Bagged GNN — ChemBERTa nodes (the pipeline `cls+mol` feature)

In [ ]:
gbres = pd.read_csv(GBCSV)
_dodge = {c: (i - (len(ORDER) - 1) / 2) * 0.55 for i, c in enumerate(ORDER)}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.ravel(), METR):
    for c in ORDER:
        sub = gbres[gbres.criterion == c]
        if sub.empty:
            continue
        qs = sorted(sub["quantile"].unique())
        means, errs = zip(*[mean_ci(sub[sub["quantile"] == q][metric].values) for q in qs])
        xd = [q + _dodge[c] for q in qs]
        st = sty(c)
        ax.plot(xd, means, ms=5, color=COL[c], label=LABEL.get(c, c), **st)
        ax.errorbar(xd, means, yerr=errs, fmt="none", ecolor=COL[c],
                    elinewidth=1.1, capsize=2, alpha=0.3, zorder=st["zorder"] - 1)
    ax.set_title(metric); ax.set_xlabel("coverage quantile")
    ax.set_ylabel(metric); ax.set_xticks(QUANTILES)
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(ORDER),
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(f"bagged GNN (n_models={N_MODELS}) - inductive - mean of seeds {SEEDS} (95% CI) - value vs quantile", y=1.06)
fig.tight_layout(); plt.show()

In [ ]:
gbres = pd.read_csv(GBCSV)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.ravel(), METR):
    r = gbres.copy()
    r["rk"] = r.groupby(["quantile", "seed"])[metric].rank(ascending=False, method="average")
    for c in ORDER:
        sub = r[r.criterion == c]
        if sub.empty:
            continue
        qs = sorted(sub["quantile"].unique())
        means = [sub[sub["quantile"] == q]["rk"].mean() for q in qs]
        ax.plot(qs, means, ms=5, color=COL[c], label=LABEL.get(c, c), **sty(c))
    ax.set_title(metric); ax.set_xlabel("coverage quantile")
    ax.set_ylabel("mean rank (1 = best)"); ax.set_xticks(QUANTILES)
    ax.invert_yaxis()
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(ORDER),
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(f"bagged GNN (n_models={N_MODELS}) - inductive - mean of seeds {SEEDS} - mean rank vs quantile", y=1.06)
fig.tight_layout(); plt.show()

## Bagged GNN — GIN nodes

In [ ]:
g2res = pd.read_csv(GBGCSV) if GBGCSV.exists() else pd.DataFrame(columns=["criterion", "quantile", "seed"])
_dodge = {c: (i - (len(ORDER) - 1) / 2) * 0.55 for i, c in enumerate(ORDER)}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.ravel(), METR):
    for c in ORDER:
        sub = g2res[g2res.criterion == c]
        if sub.empty:
            continue
        qs = sorted(sub["quantile"].unique())
        means, errs = zip(*[mean_ci(sub[sub["quantile"] == q][metric].values) for q in qs])
        xd = [q + _dodge[c] for q in qs]
        st = sty(c)
        ax.plot(xd, means, ms=5, color=COL[c], label=LABEL.get(c, c), **st)
        ax.errorbar(xd, means, yerr=errs, fmt="none", ecolor=COL[c],
                    elinewidth=1.1, capsize=2, alpha=0.3, zorder=st["zorder"] - 1)
    ax.set_title(metric); ax.set_xlabel("coverage quantile")
    ax.set_ylabel(metric); ax.set_xticks(QUANTILES)
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(ORDER),
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(f"bagged GNN GIN-nodes (n_models={N_MODELS}) - inductive - mean of seeds {SEEDS} (95% CI) - value vs quantile", y=1.06)
fig.tight_layout(); plt.show()

In [ ]:
g2res = pd.read_csv(GBGCSV) if GBGCSV.exists() else pd.DataFrame(columns=["criterion", "quantile", "seed"])
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, metric in zip(axes.ravel(), METR):
    r = g2res.copy()
    r["rk"] = r.groupby(["quantile", "seed"])[metric].rank(ascending=False, method="average")
    for c in ORDER:
        sub = r[r.criterion == c]
        if sub.empty:
            continue
        qs = sorted(sub["quantile"].unique())
        means = [sub[sub["quantile"] == q]["rk"].mean() for q in qs]
        ax.plot(qs, means, ms=5, color=COL[c], label=LABEL.get(c, c), **sty(c))
    ax.set_title(metric); ax.set_xlabel("coverage quantile")
    ax.set_ylabel("mean rank (1 = best)"); ax.set_xticks(QUANTILES)
    ax.invert_yaxis()
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(ORDER),
           frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(f"bagged GNN GIN-nodes (n_models={N_MODELS}) - inductive - mean of seeds {SEEDS} - mean rank vs quantile", y=1.06)
fig.tight_layout(); plt.show()